# Gasoline RBOB COT Mapping: CFTC → Marouen's cot_db

Goal: reconstruct the XB rows in `cot_db.csv` from public CFTC data.

Gasoline is NYMEX-only (code `111659`), no ICE leg.
- Commercial = Disaggregated Prod/Merch (Combined)
- ManagedMoney = Legacy Non-Commercial (Combined)

In [1]:
import pandas as pd

disagg = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)
legacy = pd.read_csv("../downloads/cftc/legacy_combined.csv", low_memory=False)
marouen = pd.read_csv("../cot_data/cot_db.csv")

## Step 1 — Find Gasoline RBOB in CFTC files

In [2]:
code = "111659"

for label, df in [("Disaggregated", disagg), ("Legacy", legacy)]:
    sub = df[df["cftc_contract_market_code"] == code]
    print(f"\n{label} — Code {code}:")
    print("Market and Exchange Names:")
    for n in sub["market_and_exchange_names"].unique():
        print(f"  {n}")
    if "commodity_name" in df.columns:
        print("Commodity Names:")
        for c in sub["commodity_name"].unique():
            print(f"  {c}")
    if "contract_market_name" in df.columns:
        print("Contract Market Names:")
        for cm in sub["contract_market_name"].unique():
            print(f"  {cm}")


Disaggregated — Code 111659:
Market and Exchange Names:
  GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE
  GASOLINE BLENDSTOCK (RBOB)  - NEW YORK MERCANTILE EXCHANGE
  GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE
Commodity Names:
  GASOLINE
Contract Market Names:
  GASOLINE RBOB

Legacy — Code 111659:
Market and Exchange Names:
  GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE
  GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE
Commodity Names:
  GASOLINE
Contract Market Names:
  GASOLINE RBOB


## Step 2 — Build Commercial and ManagedMoney

In [3]:
# Commercial from disaggregated combined
xb_disagg = disagg[disagg["cftc_contract_market_code"] == code].copy()
xb_disagg["date"] = pd.to_datetime(xb_disagg["report_date_as_yyyy_mm_dd"])
xb_disagg["prod_merc_positions_long"] = pd.to_numeric(xb_disagg["prod_merc_positions_long"], errors="coerce")
xb_disagg["prod_merc_positions_short"] = pd.to_numeric(xb_disagg["prod_merc_positions_short"], errors="coerce")

# ManagedMoney from legacy combined
xb_legacy = legacy[legacy["cftc_contract_market_code"] == code].copy()
xb_legacy["date"] = pd.to_datetime(xb_legacy["report_date_as_yyyy_mm_dd"])
xb_legacy["noncomm_positions_long_all"] = pd.to_numeric(xb_legacy["noncomm_positions_long_all"], errors="coerce")
xb_legacy["noncomm_positions_short_all"] = pd.to_numeric(xb_legacy["noncomm_positions_short_all"], errors="coerce")

# Merge
rebuilt = pd.merge(
    xb_disagg[["date", "prod_merc_positions_long", "prod_merc_positions_short"]],
    xb_legacy[["date", "noncomm_positions_long_all", "noncomm_positions_short_all"]],
    on="date",
)
rebuilt.columns = ["date", "CommercialLong", "CommercialShort", "MMLong", "MMShort"]
rebuilt["CommercialNet"] = rebuilt["CommercialLong"] - rebuilt["CommercialShort"]
rebuilt["MMNet"] = rebuilt["MMLong"] - rebuilt["MMShort"]

print(f"Rebuilt XB: {len(rebuilt)} rows")
rebuilt.head()

Rebuilt XB: 1033 rows


,date,CommercialLong,CommercialShort,MMLong,MMShort,CommercialNet,MMNet
0,2006-06-13,21321,40069,8772,0,-18748,8772
1,2006-06-20,22487,40127,9338,695,-17640,8643
2,2006-06-27,22786,40242,9338,199,-17456,9139
3,2006-07-03,23909,40643,8442,574,-16734,7868
4,2006-07-11,23142,40926,8205,351,-17784,7854


## Step 3 — Compare against Marouen's cot_db

In [4]:
marouen["tradeDate"] = pd.to_datetime(marouen["tradeDate"])
xb = marouen[marouen["Name"] == "XB"].dropna(subset=["Commercial_NetPosition"]).copy()

comp = pd.merge(rebuilt, xb, left_on="date", right_on="tradeDate", how="inner")
print(f"Matched rows: {len(comp)} (Marouen has {len(xb)})\n")

pairs = [
    ("CommercialLong",  "CommercialLongPosition"),
    ("CommercialShort", "CommercialShortPosition"),
    ("CommercialNet",   "Commercial_NetPosition"),
    ("MMLong",          "ManagedMoney_LongPosition"),
    ("MMShort",         "ManagedMoney_ShortPosition"),
    ("MMNet",           "ManagedMoney_NetPosition"),
]

print(f"{'rebuilt':20s}   {'marouen':30s}   {'MAPE':>8s}   {'MAE':>8s}   {'max_diff':>8s}")
print("-" * 90)
for ours, theirs in pairs:
    abs_diff = (comp[ours] - comp[theirs]).abs()
    mae = abs_diff.mean()
    denom = comp[theirs].abs().replace(0, float("nan"))
    mape = (abs_diff / denom * 100).mean()
    print(f"{ours:20s}   {theirs:30s}   {mape:7.4f}%   {mae:8.2f}   {abs_diff.max():.0f}")

Matched rows: 802 (Marouen has 809)

rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition            0.0000%       0.00   0
CommercialShort        CommercialShortPosition           0.0000%       0.00   0
CommercialNet          Commercial_NetPosition            0.0000%       0.00   0
MMLong                 ManagedMoney_LongPosition         0.0002%       0.19   1
MMShort                ManagedMoney_ShortPosition        0.0005%       0.17   1
MMNet                  ManagedMoney_NetPosition          0.0006%       0.32   2
